In [47]:
concept_path = "/mnt/abka03/concept_extraction_result/publish/gemma3n/CGDL/SNMF/coco10/train/concept/combined_concept_snmf_gl.pth"

# load past as data
import torch
data = torch.load(concept_path)
#print all they keys
print(data.keys())





dict_keys(['concepts', 'activations', 'decomposition_method', 'text_grounding', 'image_grounding_paths', 'analysis_model'])


In [48]:
import os
os.environ["TORCH_COMPILE_DISABLE"] = "1"
from huggingface_hub import login
from transformers import AutoProcessor, Gemma3nForConditionalGeneration
from PIL import Image
import requests
import os
import torch
import torch._dynamo
torch._dynamo.disable()
torch._dynamo.config.suppress_errors = True


# ✅ 1. Define your HF token (get it from https://huggingface.co/settings/tokens)
HF_TOKEN = ""  # 👈 Replace with your actual token

# ✅ 2. Set Hugging Face home directory
os.environ["HF_HOME"] = "/mnt/abka03/huggingface/hub"

# ✅ 3. Authenticate using the token (no interactive prompt)
#login(token=HF_TOKEN)

# ✅ 4. Model ID and cache path
model_id = "google/gemma-3n-E4B-it"
cache_dir = os.environ["HF_HOME"]

# ✅ 5. Load model and processor using the token
processor = AutoProcessor.from_pretrained(
    model_id,
    cache_dir=cache_dir,
)

model = Gemma3nForConditionalGeneration.from_pretrained(
    model_id,
    cache_dir=cache_dir,
    token=HF_TOKEN,
    device_map="auto",
    torch_dtype=torch.bfloat16,
).eval()





Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.44s/it]



In [ ]:
import numpy as np
# load the lm_head layer
lm_head = model.lm_head
# get the ecoder and decodr
processor = AutoProcessor.from_pretrained(
    model_id,
    cache_dir=cache_dir,
)

# create gaussian noise of shape of concepts
noise_concepts = torch.randn(data['concepts'].shape).to(dtype=torch.bfloat16)
print(noise_concepts.shape)
entropies_mean = []
entropies_sd = []
for noise_strength in torch.linspace(0.0, 1., steps=10):
    noise_concepts = noise_strength * noise_concepts +  data['concepts']
    sum_entropy = []
    for concept, label, noise  in zip(noise_concepts, data['text_grounding'], noise_concepts):
        #print(f"Concept: {label}")
        #print(concept.shape)
        logits = lm_head(concept.to(dtype=lm_head.weight.dtype))  # <--- HERE IS THE PROBLEM
        #expected mat1 and mat2 to have the same dtype, but got: float != c10::BFloat16
        logits = logits.to(dtype=torch.bfloat16)
        # apply softmax to get the probabilities
        probs = torch.softmax(logits, dim=-1)
        #label_indx = processor.encode(label, add_special_tokens=False)
        # print the max and mean value of logit
        #print(f"Max logit: {probs.max().item()}")
        #print(f"Mean logit: {probs.mean().item()}")
        # pritntop 20 tokens

        topk = torch.topk(probs, k=20, dim=-1)
        #print(f"Top 20 tokens: {topk.indices}")
        predicted_token_id = topk.indices[1].item() # get the top 1 token id
        # calcualte the enrorypy of the top 5 normilzed probabilities
        top5_probs = probs.topk(10).values
        entropy = -torch.sum(top5_probs * torch.log(top5_probs + 1e-10)).item()
        sum_entropy.append(entropy)
        #print(f"Sum entropy: {sum_entropy}")
        #print(f"Entropy: {entropy}")
        # decode the token id to token      

        #predicted_token = processor.tokenizer.decode([predicted_token_id])
        #print(f"Predicted token: {predicted_token}")
    entropies_mean.append(sum(sum_entropy) / (data['concepts'].shape[0]))
    entropies_sd.append(np.std(sum_entropy))
    print(f"Mean entropy: {sum(sum_entropy) / (data['concepts'].shape[0])}")

# experiment: faithfulness test by adding noise to concept vectors and measuring entropy of predicted tokens

import matplotlib.pyplot as plt
# plot the entropies
plt.plot(entropies_mean)
plt.fill_between(range(len(entropies_mean)), np.array(entropies_mean) - np.array(entropies_sd), np.array(entropies_mean) + np.array(entropies_sd), alpha=0.2)
plt.xlabel("Noise strength")
plt.ylabel("Mean Entropy of predicted tokens")
plt.title("Faithfulness test by adding noise to concept vectors")
plt.savefig("faithfulness_test.png")     


In [ ]:

concept_number = 2
lm_head = model.lm_head
# get the ecoder and decodr
processor = AutoProcessor.from_pretrained(
    model_id,
    cache_dir=cache_dir,
)
concept_1 = data['concepts'][concept_number]
# print concept shape if batch size bigger than i chose 1 batch
if concept_1.shape[0] > 1:
    print(concept_1.shape)

label = data['text_grounding'][concept_number]
# if label has len more than one just choose 0th
if len(label) > 1:
    label = label[0]
# create gaussian noise of shape of concepts
# set the value to vaery between 0.0 to 0.006

noise_concepts = torch.randn(concept_1.shape).to(dtype=torch.bfloat16).mul(0.0001).add(0.1).clamp(-0.01, 0.01).to(torch.bfloat16)
allProbs  = []
entropies_sd = []

for noise_strength in torch.linspace(1.0, 1.05, steps=400):
    noise_concepts = noise_strength * noise_concepts 
    concept = concept_1 +  noise_concepts
    logits = lm_head(concept.to(dtype=lm_head.weight.dtype))  # <--- HERE IS THE PROBLEM
        #expected mat1 and mat2 to have the same dtype, but got: float != c10::BFloat16
    logits = logits.to(dtype=torch.bfloat16)
        # apply softmax to get the probabilities
    probs = torch.softmax(logits, dim=-1)
    # get the max index with respect to original_concepts
    max_index = probs.argmax().item()
    label_indx = processor.tokenizer.encode(label[0], add_special_tokens=False)
    # choose 1st index if more than one
    if len(label_indx) > 1:
        label_indx = label_indx[-1]
    label_probs = probs[label_indx]
    # print label probs and convert it to cpu
    label_probs = label_probs.item() * 1e4
    # proint label probs

    allProbs.append(label_probs)
import numpy as np
import matplotlib.pyplot as plt
# plot the entropies
plt.plot(allProbs)
#plt.fill_between(range(len(entropies_mean)), np.array(entropies_mean) - np.array(entropies_sd), np.array(entropies_mean) + np.array(entropies_sd), alpha=0.2)
plt.xlabel("Noise strength")
plt.ylabel("Mean Entropy of predicted tokens")
plt.title("Faithfulness test by adding noise to concept vectors")
plt.savefig("faithfulness_test.png")   

In [ ]:

concept_number = 2
lm_head = model.lm_head
# get the ecoder and decodr
processor = AutoProcessor.from_pretrained(
    model_id,
    cache_dir=cache_dir,
)
concept_1 = data['concepts'][concept_number]
# print concept shape if batch size bigger than i chose 1 batch
if concept_1.shape[0] > 1:
    print(concept_1.shape)

label = data['text_grounding'][concept_number]
# if label has len more than one just choose 0th
if len(label) > 1:
    label = label[0]
# create gaussian noise of shape of concepts
# set the value to vaery between 0.0 to 0.006

noise_concepts = torch.randn(concept_1.shape).to(dtype=torch.bfloat16).mul(0.0001).add(0.1).clamp(-0.01, 0.01).to(torch.bfloat16)
allProbs  = []
entropies_sd = []

for noise_strength in torch.linspace(1.0, 1.05, steps=400):
    noise_concepts = noise_strength * noise_concepts 
    concept = concept_1 +  noise_concepts
    logits = lm_head(concept.to(dtype=lm_head.weight.dtype))  # <--- HERE IS THE PROBLEM
        #expected mat1 and mat2 to have the same dtype, but got: float != c10::BFloat16
    logits = logits.to(dtype=torch.bfloat16)
        # apply softmax to get the probabilities
    probs = torch.softmax(logits, dim=-1)
    # get the max index with respect to original_concepts
    max_index = probs.argmax().item()
    label_indx = processor.tokenizer.encode(label[0], add_special_tokens=False)
    # choose 1st index if more than one
    if len(label_indx) > 1:
        label_indx = label_indx[-1]
    label_probs = probs[label_indx]
    # print label probs and convert it to cpu
    label_probs = label_probs.item() * 1e4
    # proint label probs

    allProbs.append(label_probs)
import numpy as np
import matplotlib.pyplot as plt
# plot the entropies
plt.plot(allProbs)
#plt.fill_between(range(len(entropies_mean)), np.array(entropies_mean) - np.array(entropies_sd), np.array(entropies_mean) + np.array(entropies_sd), alpha=0.2)
plt.xlabel("Noise strength")
plt.ylabel("Mean Entropy of predicted tokens")
plt.title("Faithfulness test by adding noise to concept vectors")
plt.savefig("faithfulness_test.png")   

In [ ]:
# Plot label-token probability curves for concepts (deterministic fixed perturbation)
import matplotlib.pyplot as plt
import numpy as np
import torch

# Resolve a usable device (avoid meta tensors)
if 'lm_head' not in globals():
    lm_head = model.lm_head

# Find a real (non-meta) parameter to infer execution device
def _find_real_device(m):
    for p in m.parameters():
        if p.device.type != 'meta':
            return p.device
    for b in m.buffers():
        if b.device.type != 'meta':
            return b.device
    return torch.device('cpu')

exec_device = _find_real_device(model)
if lm_head.weight.device.type == 'meta':
    lm_head = lm_head.to(exec_device)

concept_tensor = data['concepts']  # shape: (num_concepts, hidden_dim)
text_labels = data.get('text_grounding', [])
num_available = concept_tensor.shape[0]

concept_indices = list(range(min(10, num_available)))  # adjust count as needed
print(f'Generating curves for concept indices: {concept_indices} on device {exec_device}')

# Noise / perturbation config (deterministic)
noise_strengths = torch.linspace(0.0, 1.0, steps=80, device=exec_device)
fixed_delta_value = 0.01  # constant additive value per dimension
curves = {}

with torch.no_grad():
    concepts_exec = concept_tensor.to(device=exec_device, dtype=lm_head.weight.dtype)

    for idx in concept_indices:
        base_vec = concepts_exec[idx]  # (hidden_dim,)

        # Deterministic fixed direction: all ones scaled (could also use sign(base_vec))
        # Using ones is purely structural; it measures uniform sensitivity.
        noise_base = torch.ones_like(base_vec) * fixed_delta_value
        # Optional alternative (uncomment to use sign-based direction):
        # noise_base = base_vec.sign() * fixed_delta_value

        # Resolve label token id
        label_raw = text_labels[idx] if idx < len(text_labels) else ''
        if isinstance(label_raw, (list, tuple)):
            label_raw = label_raw[0] if len(label_raw) > 0 else ''
        token_ids = processor.tokenizer.encode(label_raw, add_special_tokens=False)
        label_token_id = token_ids[-1] if len(token_ids) > 0 else 0

        probs_curve = []
        for ns in noise_strengths:
            perturbed_vec = base_vec + ns * noise_base
            logits = lm_head(perturbed_vec.unsqueeze(0))  # (1, vocab)
            probs = torch.softmax(logits.float(), dim=-1)
            p_label = probs[0, label_token_id].item()
            probs_curve.append(p_label)

        arr = np.array(probs_curve, dtype=np.float64)
        # Keep raw (not shape-normalized); if normalization desired, uncomment:
        denom = arr.max() - arr.min(); arr = np.zeros_like(arr) if denom < 1e-12 else (arr - arr.min()) / denom
        curves[f'Concept {idx+1}'] = arr
noise_cpu = noise_strengths.cpu().numpy()
plt.figure(figsize=(9,5))
for name, curve in curves.items():
    plt.plot(noise_cpu, curve, label=name)

plt.xlabel('Perturbation strength (scaled fixed delta)')
plt.ylabel('Label token probability')
plt.title('Concept sensitivity (deterministic fixed perturbation)')
plt.legend(ncol=2, fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [50]:
import torch
from transformers import AutoProcessor
concept_number = 5
# --- prep ---
lm_head = model.lm_head
processor = AutoProcessor.from_pretrained(model_id, cache_dir=cache_dir)
concept_1 = data['concepts'][concept_number]

# pick the label token id
label = data['text_grounding'][concept_number]
if isinstance(label, (list, tuple)) and len(label) > 1:
    label = label[0]
label_ids = processor.tokenizer.encode(label[0], add_special_tokens=False)
label_id = label_ids[-1] if len(label_ids) > 1 else label_ids[0]

# --- enable gradient tracking on the concept vector ---
concept_1 = concept_1.detach().clone().requires_grad_(True)

# --- forward pass ---
logits = lm_head(concept_1.to(dtype=lm_head.weight.dtype, device=lm_head.weight.device))  # shape: (vocab_size,)
probs  = torch.softmax(logits, dim=-1)

# probability of the target token
target_prob = probs[label_id]

# --- backward pass ---
# you can use log-prob or prob itself; here we use negative log for clarity
loss = -torch.log(target_prob + 1e-12)
loss.backward()

# --- gradient is now stored in concept_1.grad ---
grad_concept = concept_1.grad        # same shape as concept_1
print("Gradient shape:", grad_concept.shape)
print("Gradient (sample):", grad_concept[ :5])

Gradient shape: torch.Size([2048])
Gradient (sample): tensor([-0.0269,  0.0071, -0.0081, -0.0295,  0.0130])
